# does wikipedia attention predict how wild a stock gets?

basically: before a stock has a weirdly wild week, are people suddenly reading about it on wikipedia? i pulled daily wikipedia pageviews for a bunch of big companies and compared them to stock volatility. the big question is whether wikipedia helps predict the next week after we already account for what the stock has been doing recently.

the fair test is simple: if wikipedia adds anything, it should beat a model that already knows the stock's recent volatility. if it does not, then wikipedia is probably just reacting to news that already hit the market.

**contents**
1. setup
2. get the data
3. look at the data
4. build the table (features and what we predict)
5. does a surge in views matter? (the simple check)
6. models (simple linear regression)
7. results
8. notes to write up

## 1. setup

uncomment the pip line if you're missing anything. if the dropdowns don't show up in vs code, make sure `ipywidgets` is installed and reload the window.

In [17]:
# %pip install pandas numpy requests yfinance plotly ipywidgets scikit-learn

import time
import warnings
from datetime import date
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

START = "2017-01-01"
END = date.today().isoformat()

## 2. get the data

two free sources, no api keys:
- **stock prices** from yahoo finance (the `yfinance` package). we need open, high, low, close and volume for each day
- **wikipedia pageviews**: how many times each company's english wikipedia page was read each day (wikimedia has a free api for this)

the wikipedia page titles have to be exact. if one fails the code prints which one, fix the title and delete `data/wiki_views.csv` to try again.

everything is saved to the `data/` folder so we only download once. put that folder on github so the repo has the data in it.

In [18]:
# ticker : wikipedia article title
UNIVERSE = {
    "AAPL": "Apple_Inc.",
    "MSFT": "Microsoft",
    "AMZN": "Amazon_(company)",
    "GOOGL": "Alphabet_Inc.",
    "META": "Meta_Platforms",
    "TSLA": "Tesla,_Inc.",
    "NVDA": "Nvidia",
    "NFLX": "Netflix",
    "AMD": "Advanced_Micro_Devices",
    "JPM": "JPMorgan_Chase",
    "XOM": "ExxonMobil",
    "BA": "Boeing",
    "DIS": "The_Walt_Disney_Company",
    "KO": "The_Coca-Cola_Company",
    "WMT": "Walmart",
}

pd.DataFrame({"ticker": list(UNIVERSE), "wiki_article": list(UNIVERSE.values())}).to_csv(
    DATA_DIR / "universe.csv", index=False
)

In [19]:
# stock prices
PRICE_FILE = DATA_DIR / "prices.csv"

if PRICE_FILE.exists():
    prices = pd.read_csv(PRICE_FILE, parse_dates=["date"])
else:
    raw = yf.download(list(UNIVERSE), start=START, end=END, auto_adjust=True,
                      progress=False, group_by="ticker")
    parts = []
    for t in UNIVERSE:
        try:
            one = raw[t][["Open", "High", "Low", "Close", "Volume"]].dropna().copy()
        except KeyError:
            print("no price data for", t)
            continue
        one["ticker"] = t
        one.index.name = "date"
        parts.append(one.reset_index())
    prices = pd.concat(parts, ignore_index=True)
    prices.to_csv(PRICE_FILE, index=False)

print(prices.shape)
prices.head()

(37080, 7)


Price,date,Open,High,Low,Close,Volume,ticker
0,2017-01-03,26.617767,26.739592,26.378712,26.698217,115127600,AAPL
1,2017-01-04,26.629255,26.780964,26.606270,26.668331,84472400,AAPL
2,2017-01-05,26.645340,26.861409,26.620055,26.803944,88774400,AAPL
3,2017-01-06,26.843025,27.160233,26.771769,27.102768,127007600,AAPL
4,2017-01-09,27.111957,27.452150,27.109660,27.351011,134247600,AAPL


In [20]:
# wikipedia pageviews
WIKI_FILE = DATA_DIR / "wiki_views.csv"
# wikimedia asks for a real contact in the user agent, put your own email here
HEADERS = {"User-Agent": "student-project-wiki-stocks/0.1 (your-email@example.com)"}

def get_views(article):
    end = date.today().strftime("%Y%m%d")
    url = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
           f"en.wikipedia/all-access/user/{quote(article, safe='')}/daily/20170101/{end}")
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        print("failed:", article, r.status_code)
        return None
    df = pd.DataFrame(r.json()["items"])
    df["date"] = pd.to_datetime(df["timestamp"].str[:8], format="%Y%m%d")
    return df[["date", "views"]]

if WIKI_FILE.exists():
    wiki = pd.read_csv(WIKI_FILE, parse_dates=["date"])
else:
    parts = []
    for ticker, article in UNIVERSE.items():
        v = get_views(article)
        if v is not None:
            v["ticker"] = ticker
            parts.append(v)
        time.sleep(0.3)  # don't spam the api
    wiki = pd.concat(parts, ignore_index=True)
    wiki.to_csv(WIKI_FILE, index=False)

print(wiki.shape)
wiki.head()

(52055, 3)


,date,views,ticker
0,2017-01-01,8112,AAPL
1,2017-01-02,11006,AAPL
2,2017-01-03,12029,AAPL
3,2017-01-04,12570,AAPL
4,2017-01-05,12288,AAPL


## 3. look at the data

pick a stock below. top is the price, middle is the **daily range** (how far the price moved in a day), bottom is wikipedia views. use the slider under the charts to zoom in.

the daily range is just `(high - low) / close`. so 0.03 means the price moved about 3% from the lowest to the highest point that day. it's a simple way to measure how wild a day was.

In [21]:
tickers_ok = sorted(set(prices.ticker) & set(wiki.ticker))

def show_stock(ticker="NVDA", smooth_days=10):
    p = prices[prices.ticker == ticker].sort_values("date").set_index("date")
    w = wiki[wiki.ticker == ticker].set_index("date")["views"]
    day_range = (p["High"] - p["Low"]) / p["Close"]
    day_range_smooth = day_range.rolling(smooth_days).mean()

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=("price", f"daily range ({smooth_days} day average)", "wikipedia views per day"))
    fig.add_trace(go.Scatter(x=p.index, y=p["Close"], line=dict(width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=day_range_smooth.index, y=day_range_smooth, line=dict(width=1, color="crimson")), row=2, col=1)
    fig.add_trace(go.Scatter(x=w.index, y=w, line=dict(width=1, color="seagreen")), row=3, col=1)
    fig.update_yaxes(tickformat=".1%", row=2, col=1)
    fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.05), row=3, col=1)
    fig.update_layout(height=700, showlegend=False, title=ticker)
    fig.show()

widgets.interact(
    show_stock,
    ticker=widgets.Dropdown(options=tickers_ok, value="NVDA" if "NVDA" in tickers_ok else tickers_ok[0]),
    smooth_days=widgets.IntSlider(value=10, min=1, max=60),
);

interactive(children=(Dropdown(description='ticker', index=11, options=('AAPL', 'AMD', 'AMZN', 'BA', 'DIS', 'G…

## 4. build the table

one row per stock per day. here's every column:

- `day_range`: how much the price moved that day, `(high - low) / close`
- `past_week`: average daily range over the last 5 trading days
- `past_month`: average daily range over the last 22 trading days
- `wiki_ratio`: average views over the last 7 days divided by the average over the last 90 days. so 1.0 is normal, 2.0 means the page is getting double its usual attention
- `next_week`: average daily range over the **next** 5 trading days. **this is what we're trying to predict**

the `shift(-5)` line is the one that looks into the future, and it's only used to make the thing we predict. the other columns only use stuff up to today.

In [22]:
def build_table():
    pieces = []
    for t in tickers_ok:
        p = prices[prices.ticker == t].sort_values("date").set_index("date")
        w = wiki[wiki.ticker == t].drop_duplicates("date").set_index("date")["views"]

        # wikipedia has every calendar day (weekends too) so do the averages on that first
        w = w.asfreq("D").ffill()
        ratio = w.rolling(7).mean() / w.rolling(90).mean()

        out = pd.DataFrame(index=p.index)
        out["ticker"] = t
        out["day_range"] = (p["High"] - p["Low"]) / p["Close"]
        out["past_week"] = out["day_range"].rolling(5).mean()
        out["past_month"] = out["day_range"].rolling(22).mean()
        out["wiki_ratio"] = ratio.reindex(out.index)
        # average of the next 5 days (rolling mean ends on day t+5, shift pulls it back to day t)
        out["next_week"] = out["day_range"].rolling(5).mean().shift(-5)
        pieces.append(out)

    table = pd.concat(pieces).reset_index().rename(columns={"index": "date"})
    return table.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

table = build_table()
print(table.shape, "rows")
table.head()

(34856, 7) rows


,date,ticker,day_range,past_week,past_month,wiki_ratio,next_week
0,2017-03-31,AAPL,0.008771,0.013395,0.010226,0.935723,0.008916
1,2017-04-03,AAPL,0.007446,0.011194,0.010068,0.948793,0.008796
2,2017-04-04,AAPL,0.011881,0.008813,0.010204,0.951994,0.011066
3,2017-04-05,AAPL,0.011457,0.009300,0.010344,0.944242,0.010382
4,2017-04-06,AAPL,0.007448,0.009401,0.010294,0.933297,0.010778


In [23]:
table.describe().round(4)

,date,day_range,past_week,past_month,wiki_ratio,next_week
count,34856,34856.0000,34856.0000,34856.0000,34856.0000,34856.0000
mean,2022-02-10 21:49:07,0.0257,0.0257,0.0257,1.0035,0.0257
min,2017-03-31 00:00:00,0.0030,0.0048,0.0064,0.2289,0.0048
25%,2019-09-23 00:00:00,0.0143,0.0159,0.0167,0.8894,0.0160
50%,2022-03-10 00:00:00,0.0211,0.0222,0.0225,0.9707,0.0222
75%,2024-07-03 00:00:00,0.0316,0.0313,0.0312,1.0574,0.0314
max,2026-10-26 00:00:00,0.2715,0.1960,0.1417,12.8183,0.1960
std,NaN,0.0174,0.0143,0.0126,0.3247,0.0143


## 5. does a surge in views matter? (the simple check)

no models yet, just a straight comparison. i sort every day into 5 groups (quintiles) by `wiki_ratio`, from the quietest 20% of days to the 20% with the biggest attention jump.

then for each group i look at how much **more** (or less) volatile the next week was compared to the last week:

`change = next_week / past_week - 1`

i use the median because a few huge days can mess up an average. if wikipedia attention really warns us about wild weeks, the right hand bars should be higher.

In [24]:
table["change"] = table["next_week"] / table["past_week"] - 1
table["attention_group"] = pd.qcut(table["wiki_ratio"], 5,
                                   labels=["1 (quietest)", "2", "3", "4", "5 (biggest jump)"])

by_group = table.groupby("attention_group", observed=True)["change"].median()

fig = px.bar(x=by_group.index.astype(str), y=by_group.values,
             title="median change in volatility next week, by how much attention jumped")
fig.add_hline(y=0, line_dash="dot")
fig.update_layout(xaxis_title="wiki attention group", yaxis_title="next week vs last week", yaxis_tickformat=".0%", height=400)
fig.show()

In [25]:
# scatter of every day (random sample so it doesn't lag), x axis cut off so a few crazy values don't squash everything
sample = table.sample(min(6000, len(table)), random_state=1)
sample = sample[(sample.wiki_ratio < 4) & (sample.change < 2)]

fig = px.scatter(sample, x="wiki_ratio", y="change", color="ticker", opacity=0.4,
                 title="attention jump vs change in volatility (each dot is one stock on one day)")
fig.update_layout(height=480, yaxis_tickformat=".0%", xaxis_title="wiki ratio (1 = normal)", yaxis_title="next week vs last week")
fig.show()

the bar chart is the quick check. if the bars rise pretty steadily from group 1 to group 5, that means days with a big wikipedia spike are followed by more volatile weeks. if the groups are all basically flat, then wikipedia is not telling us much. in this setup, the pattern is weak, so it does not look like a strong early warning signal.


## 6. models

now a proper test. i split by date:
- **train**: everything before 2023 (the model learns from this)
- **test**: 2023 onwards (the model has never seen it, we check how well it predicts)

we never shuffle the dates. if we mixed them up we'd be predicting the past using the future, which is cheating.

three things to compare:
1. **"same as last week"**: no model, just guess next week = the last 5 days. this is the dumb benchmark
2. **linear regression with past volatility only**: uses `past_week` and `past_month`
3. **linear regression with past volatility and wikipedia**: same thing but also `wiki_ratio`

linear regression just finds the best straight line formula to predict next week from the inputs. if model 3 beats model 2, wikipedia added something.

scores:
- **MAE** (mean absolute error): on average how far off are we. lower is better
- **R2**: how much of the ups and downs we explain. 1 is perfect, 0 is no better than guessing the average, and it can go negative

In [26]:
SPLIT_DATE = "2023-01-01"

train = table[table.date < SPLIT_DATE]
test = table[table.date >= SPLIT_DATE].copy()
print(len(train), "train rows,", len(test), "test rows")

feature_sets = {
    "past volatility only": ["past_week", "past_month"],
    "past volatility + wikipedia": ["past_week", "past_month", "wiki_ratio"],
}

# model 1: no model, just copy last week
test["pred_same_as_last_week"] = test["past_week"]

# models 2 and 3
for name, cols in feature_sets.items():
    model = LinearRegression()
    model.fit(train[cols], train["next_week"])
    test["pred_" + name] = model.predict(test[cols])

20501 train rows, 14355 test rows


## 7. results

In [27]:
rows = []
for label, col in [("same as last week", "pred_same_as_last_week"),
                   ("past volatility only", "pred_past volatility only"),
                   ("past volatility + wikipedia", "pred_past volatility + wikipedia")]:
    rows.append({
        "model": label,
        "MAE": mean_absolute_error(test["next_week"], test[col]),
        "R2": r2_score(test["next_week"], test[col]),
    })
results = pd.DataFrame(rows).set_index("model")
results.round(5)

,MAE,R2
model,,
same as last week,0.00589,0.39532
past volatility only,0.00526,0.53805
past volatility + wikipedia,0.00526,0.53805


In [28]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("MAE (lower is better)", "R2 (higher is better)"))
fig.add_trace(go.Bar(x=results.index, y=results["MAE"], marker_color="steelblue"), row=1, col=1)
fig.add_trace(go.Bar(x=results.index, y=results["R2"], marker_color="seagreen"), row=1, col=2)
fig.update_layout(showlegend=False, height=420)
fig.show()

In [29]:
# does the wikipedia model win for each stock? positive = wikipedia version has lower error
rows = []
for t, d in test.groupby("ticker"):
    mae_without = mean_absolute_error(d["next_week"], d["pred_past volatility only"])
    mae_with = mean_absolute_error(d["next_week"], d["pred_past volatility + wikipedia"])
    rows.append({"ticker": t, "improvement from wikipedia": (mae_without - mae_with) / mae_without})
per_stock = pd.DataFrame(rows).sort_values("improvement from wikipedia")

fig = px.bar(per_stock, x="improvement from wikipedia", y="ticker", orientation="h",
             title="how much lower is the error when we add wikipedia? (per stock)")
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(height=500, xaxis_tickformat=".1%")
fig.show()

In [30]:
# actual vs predicted for one stock in the test period
def show_predictions(ticker="TSLA"):
    d = test[test.ticker == ticker].sort_values("date")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d.date, y=d["next_week"], name="actual", line=dict(color="black", width=1.5)))
    fig.add_trace(go.Scatter(x=d.date, y=d["pred_same_as_last_week"], name="same as last week", line=dict(width=1)))
    fig.add_trace(go.Scatter(x=d.date, y=d["pred_past volatility + wikipedia"], name="model with wikipedia",
                             line=dict(width=1, color="crimson")))
    fig.update_layout(title=f"{ticker}: average daily range over the next 5 days", yaxis_tickformat=".1%",
                      height=450, xaxis=dict(rangeslider=dict(visible=True, thickness=0.05)))
    fig.show()

widgets.interact(show_predictions,
                 ticker=widgets.Dropdown(options=sorted(test.ticker.unique()),
                                         value="TSLA" if "TSLA" in set(test.ticker) else test.ticker.iloc[0]));

interactive(children=(Dropdown(description='ticker', index=12, options=('AAPL', 'AMD', 'AMZN', 'BA', 'DIS', 'G…

the results are pretty clear: wikipedia did not help once past volatility was already in the model.

| model | MAE | R2 |
| --- | ---: | ---: |
| same as last week | 0.00589 | 0.39532 |
| past volatility only | 0.00526 | 0.53805 |
| past volatility + wikipedia | 0.00526 | 0.53805 |

so the model with wikipedia was basically identical to the past-volatility-only model. the simple benchmark was worse, which means recent volatility matters, but wikipedia did not add any real predictive power. in plain English: the extra pageview signal is not giving us a useful edge.


## 8. notes to write up

- **what i found**: once recent volatility was included, adding wikipedia pageviews did not improve the forecast. the MAE and R2 were basically the same, so there was no real extra signal.
- **why it might be like that**: wikipedia traffic often spikes after the news has already broken, so it may just be a reaction to what happened instead of a warning sign. a lot of the attention is also tied to earnings, product news, and big company events that already show up in recent volatility.
- **limits**: only 15 big companies, one date-based train/test split, and a fairly simple linear model. that is enough for a first pass, but not enough to claim a strong effect.
- **next things to try**:
  - add earnings dates and see if wikipedia still helps
  - try different split dates to check if the result holds up
  - compare with google trends or reddit attention
  - try a more flexible model like a random forest or gradient boosting
